OpenSMILE, high gaussian noise 

In [20]:
#load 
from pathlib import Path
import subprocess
import pandas as pd
import tempfile
import os

In [21]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_gaussian_noise_perturbations_pain" / "high_gaussian_noise"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_high_gaussian_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_high_gaussian_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Kolommen netjes ordenen
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,33.82516,0.126939,32.61083,34.10638,37.14517,...,-0.047993,0.008274,0.082044,3.383459,2.298851,0.100000,0.041231,0.268571,0.296380,-36.62595
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.59982,0.057266,35.03291,35.39471,38.89122,...,-0.047717,0.012750,0.080412,3.773585,2.898551,0.091667,0.033375,0.233333,0.230338,-35.82957
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.37175,0.038957,33.24994,34.01749,35.39640,...,-0.050461,0.011694,0.078031,3.083700,2.252252,0.132000,0.090200,0.241667,0.377642,-36.31226
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,33.92975,0.130533,33.56673,34.32417,36.76130,...,-0.041646,0.006949,0.075227,4.464286,3.652968,0.085000,0.071589,0.191429,0.214371,-37.15251
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,33.52404,0.177534,33.14048,34.08512,37.20667,...,-0.049143,0.011133,0.074970,2.542373,3.030303,0.092857,0.047423,0.214286,0.265538,-35.62039


OpenSMILE, low gaussian noise 

In [22]:
# Path
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_gaussian_noise_perturbations_pain" / "low_gaussian_noise"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_low_gaussian_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_low_gaussian_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# Columns
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,30.90117,0.256540,30.69261,33.91220,35.34599,...,-0.049006,-0.001545,0.083855,3.759399,3.065134,0.088750,0.044564,0.192222,0.162123,-36.63449
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56697,0.143086,35.00083,35.37520,38.83775,...,-0.048778,0.003345,0.080464,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-35.83871
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.36370,0.038547,33.35050,34.00207,35.33662,...,-0.053167,0.002206,0.078596,3.083700,2.252252,0.138000,0.091957,0.236667,0.375751,-36.32599
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,29.84299,0.295013,14.93718,34.11245,36.57940,...,-0.041269,-0.001047,0.078764,4.464286,4.566210,0.085000,0.065000,0.125555,0.134504,-37.15867
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.55619,0.217961,32.86898,33.92832,37.18552,...,-0.050745,0.002613,0.074903,2.966102,3.463203,0.086250,0.048974,0.180000,0.257196,-35.63274


OpenSMILE, medium gaussian noise 

In [23]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_gaussian_noise_perturbations_pain" / "medium_gaussian_noise"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_medium_gaussian_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_medium_gaussian_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

# columns
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,31.11563,0.249000,32.40583,33.94053,35.34982,...,-0.048942,0.003062,0.083708,3.759399,3.065134,0.087500,0.046301,0.193333,0.164114,-36.63192
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,35.56597,0.143191,35.00144,35.37555,38.83817,...,-0.048499,0.008989,0.080519,3.773585,3.381643,0.081429,0.039795,0.194286,0.175650,-35.83884
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.37297,0.038711,33.31701,34.01583,35.35612,...,-0.052465,0.006585,0.078444,3.083700,2.252252,0.136000,0.094149,0.238333,0.379448,-36.32253
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,30.69821,0.268832,23.22491,34.18171,36.66275,...,-0.041163,0.003565,0.077175,4.464286,4.566210,0.080000,0.065422,0.119000,0.130342,-37.15673
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,32.55663,0.217970,32.86958,33.93191,37.18370,...,-0.050805,0.008056,0.074926,2.966102,3.463203,0.086250,0.048974,0.180000,0.257196,-35.63209


OpenSMILE, very high gaussian noise 

In [24]:
# paths
BASE_DIR = Path(r"C:\Users\marti\Documents\Technical Medicine Master\Stages TM2\TM2-3\Technische opdracht")

OPENSMILE_EXE = BASE_DIR / "tools" / "openSMILE" / "bin" / "SMILExtract.exe"
CONFIG_FILE = BASE_DIR / "tools" / "openSMILE" / "config" / "egemaps" / "v02" / "eGeMAPSv02.conf"

AUDIO_ROOT = BASE_DIR / "TAME" / "wiener_gaussian_noise_perturbations_pain" / "very_high_gaussian_noise"

OUTPUT_FEATURES_CSV = BASE_DIR / "opensmile_very_high_gaussian_features_pain.csv"
OUTPUT_FAILED_CSV = BASE_DIR / "opensmile_very_high_gaussian_failed_files.csv"

# check
print("SMILExtract exists:", OPENSMILE_EXE.exists())
print("Config exists:", CONFIG_FILE.exists())
print("Audio root exists:", AUDIO_ROOT.exists())

if not OPENSMILE_EXE.exists():
    raise FileNotFoundError(f"SMILExtract.exe not found:\n{OPENSMILE_EXE}")

if not CONFIG_FILE.exists():
    raise FileNotFoundError(f"Config file not found:\n{CONFIG_FILE}")

if not AUDIO_ROOT.exists():
    raise FileNotFoundError(f"Audio root not found:\n{AUDIO_ROOT}")

#WAV files
wav_files = sorted(AUDIO_ROOT.rglob("*.wav"))
print(f"Number of wav. files found: {len(wav_files)}")

if len(wav_files) == 0:
    raise ValueError("no .wav files found.")

# Run OpenSMILE
all_rows = []
failed_files = []

for i, wav_file in enumerate(wav_files, start=1):
    print(f"[{i}/{len(wav_files)}] Processing: {wav_file.name}")

    participant_id = wav_file.parent.name
    filename = wav_file.name

    with tempfile.NamedTemporaryFile(suffix=".csv", delete=False) as tmp:
        temp_output_csv = Path(tmp.name)

    try:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)

        cmd = [
            str(OPENSMILE_EXE),
            "-C", str(CONFIG_FILE),
            "-I", str(wav_file),
            "-csvoutput", str(temp_output_csv),
        ]

        result = subprocess.run(cmd, capture_output=True, text=True)

        if result.returncode != 0:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "openSMILE return code != 0",
                "stderr": result.stderr
            })
            continue

        if not temp_output_csv.exists():
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "no output made",
                "stderr": result.stderr
            })
            continue

        try:
            feature_df = pd.read_csv(temp_output_csv, sep=";", engine="python")
        except Exception as e:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": f"error in csv: {e}",
                "stderr": result.stderr
            })
            continue

        if feature_df.empty:
            failed_files.append({
                "participant_id": participant_id,
                "filename": filename,
                "file_path": str(wav_file),
                "reason": "Output csv is leeg",
                "stderr": result.stderr
            })
            continue

        #adding information 
        row = feature_df.iloc[0].to_dict()
        row["name"] = filename
        row["participant_id"] = participant_id
        row["filename"] = filename
        row["file_path"] = str(wav_file)

        all_rows.append(row)

    except Exception as e:
        failed_files.append({
            "participant_id": participant_id,
            "filename": filename,
            "file_path": str(wav_file),
            "reason": f"error {e}",
            "stderr": ""
        })

    finally:
        if temp_output_csv.exists():
            temp_output_csv.unlink(missing_ok=True)


#Save
features_df = pd.DataFrame(all_rows)

#ordering columns
preferred_front_cols = ["participant_id", "filename", "file_path", "name", "frameTime"]
remaining_cols = [col for col in features_df.columns if col not in preferred_front_cols]
features_df = features_df[[col for col in preferred_front_cols if col in features_df.columns] + remaining_cols]

features_df.to_csv(OUTPUT_FEATURES_CSV, index=False)

print("\nKlaar.")
print("number of errors in features:", len(features_df))
print("numer of errors in the files:", len(failed_files))
print("Features saves in :")
print(OUTPUT_FEATURES_CSV)

if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(OUTPUT_FAILED_CSV, index=False)
    print("\n errors saved:")
    print(OUTPUT_FAILED_CSV)


# Results
print("\nShape features dataframe:", features_df.shape)
display(features_df.head())

SMILExtract exists: True
Config exists: True
Audio root exists: True
Number of wav. files found: 1055
[1/1055] Processing: p10085.LC.1.161.wav
[2/1055] Processing: p10085.LC.12.59.wav
[3/1055] Processing: p10085.LC.14.45.wav
[4/1055] Processing: p10085.LC.18.41.wav
[5/1055] Processing: p10085.LC.2.106.wav
[6/1055] Processing: p10085.LC.23.156.wav
[7/1055] Processing: p10085.LC.24.172.wav
[8/1055] Processing: p10085.LC.26.99999.wav
[9/1055] Processing: p10085.LC.3.60.wav
[10/1055] Processing: p10085.LC.30.132.wav
[11/1055] Processing: p10085.LC.31.99999.wav
[12/1055] Processing: p10085.LC.33.6.wav
[13/1055] Processing: p10085.LC.34.122.wav
[14/1055] Processing: p10085.LC.37.71.wav
[15/1055] Processing: p10085.LC.38.160.wav
[16/1055] Processing: p10085.LC.4.76.wav
[17/1055] Processing: p10085.LC.40.174.wav
[18/1055] Processing: p10085.LC.41.99999.wav
[19/1055] Processing: p10085.LC.5.134.wav
[20/1055] Processing: p10085.LC.6.99999.wav
[21/1055] Processing: p10085.LC.7.62.wav
[22/1055] Pr

,participant_id,filename,file_path,name,frameTime,F0semitoneFrom27.5Hz_sma3nz_amean,F0semitoneFrom27.5Hz_sma3nz_stddevNorm,F0semitoneFrom27.5Hz_sma3nz_percentile20.0,F0semitoneFrom27.5Hz_sma3nz_percentile50.0,F0semitoneFrom27.5Hz_sma3nz_percentile80.0,...,slopeUV0-500_sma3nz_amean,slopeUV500-1500_sma3nz_amean,spectralFluxUV_sma3nz_amean,loudnessPeaksPerSec,VoicedSegmentsPerSec,MeanVoicedSegmentLengthSec,StddevVoicedSegmentLengthSec,MeanUnvoicedSegmentLength,StddevUnvoicedSegmentLength,equivalentSoundLevel_dBp
0,p10085,p10085.LC.1.161.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.1.161.wav,0.0,34.90938,0.057080,33.32947,34.27920,37.17465,...,-0.045610,0.009774,0.085643,3.383459,2.298851,0.086667,0.032489,0.280000,0.308730,-36.58761
1,p10085,p10085.LC.12.59.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.12.59.wav,0.0,36.57626,0.057589,35.02851,35.38795,38.90328,...,-0.044731,0.012113,0.082540,3.301887,2.898551,0.090000,0.033665,0.235000,0.229111,-35.80139
2,p10085,p10085.LC.14.45.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.14.45.wav,0.0,34.37094,0.039022,33.25134,34.01879,35.39931,...,-0.047939,0.013330,0.078813,3.083700,2.252252,0.132000,0.090200,0.241667,0.377642,-36.28146
3,p10085,p10085.LC.18.41.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.18.41.wav,0.0,35.07177,0.039629,33.91322,34.42411,36.82372,...,-0.038430,0.008621,0.075678,4.464286,4.109589,0.066667,0.051640,0.198571,0.227623,-37.11533
4,p10085,p10085.LC.2.106.wav,C:\Users\marti\Documents\Technical Medicine Ma...,p10085.LC.2.106.wav,0.0,34.05785,0.151141,33.23326,34.14742,37.23193,...,-0.046168,0.012770,0.076239,2.542373,3.030303,0.088571,0.051666,0.192500,0.255575,-35.59097
